In [1]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os  

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
date = "2025-12"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

In [15]:
end_use = "naics?get=CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url) 
    
print(r)
    
df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

df.columns = r.json()[0]

df["total_imports"] = df["CON_VAL_MO"].astype(float)

df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])

top_products = grp.agg({"total_imports":"sum","CTY_CODE":"first"})

country_list = list(top_products.sort_values(by = "total_imports", ascending = False).CTY_CODE)[0:31]


['TOTAL FOR ALL COUNTRIES','NAFTA','EUROPEAN UNION']

<Response [200]>


['TOTAL FOR ALL COUNTRIES', 'NAFTA', 'EUROPEAN UNION']

In [16]:
df.tail()

,CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL,time,total_imports
39049,13896714,7940,ZAMBIA,DET,2025-11,13896714.0
39050,12624834,7950,ESWATINI,DET,2025-11,12624834.0
39051,15520151,7960,ZIMBABWE,DET,2025-11,15520151.0
39052,2368430,7970,MALAWI,DET,2025-11,2368430.0
39053,8270767,7990,LESOTHO,DET,2025-11,8270767.0


In [17]:
df.time.unique()[-1]

'2025-11'

In [3]:
country_list = [""]

In [7]:
end_use = "naics?get=CTY_NAME,CON_VAL_MO,CAL_DUT_MO,NAICS,NAICS_SDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

surl  = surl + my_key + "&time=" + "from+2020-01" + "&COMM_LVL=NA6" 

for xxx in country_list:
    
    out_file = ".\\data"+ "\\imports\\" + xxx + "naics-data-" + date + ".parquet"
    
    if xxx == "":
        out_file = ".\\data"+ "\\imports\\" + "TOTAL" + "naics-data-" + date + ".parquet"
    
    
    if os.path.exists(out_file):
        
        print("Already have downloaded file")
        
        continue
    
    print(xxx)
    
    url = surl + "&CTY_CODE=" + xxx
    
    if xxx == "":
        url = surl
    
    r = requests.get(url) 
    
    print(r)
    
    foo = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

    foo.columns = r.json()[0]

    pq.write_table(pa.Table.from_pandas(foo), out_file)


<Response [200]>


In [9]:
foo.head()

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,NAICS,NAICS_SDESC,time,COMM_LVL
0,TOTAL FOR ALL COUNTRIES,17385481,52327,111110,SOYBEANS,2020-01,NA6
1,TOTAL FOR ALL COUNTRIES,63804211,470841,111120,OILSEEDS (EXCEPT SOYBEAN),2020-01,NA6
2,TOTAL FOR ALL COUNTRIES,37535042,527655,111130,DRY PEAS & BEANS,2020-01,NA6
3,TOTAL FOR ALL COUNTRIES,56326683,184568,111140,WHEAT,2020-01,NA6
4,TOTAL FOR ALL COUNTRIES,24800161,5228,111150,CORN,2020-01,NA6


In [20]:
def load_naics_imports(parquet_path):
    df = pq.read_table(parquet_path).to_pandas()
    df['imports'] = pd.to_numeric(df['CON_VAL_MO'], errors='coerce')
    df['duties']  = pd.to_numeric(df['CAL_DUT_MO'],  errors='coerce')
    df['time']    = pd.to_datetime(df['time'])
    df['naics6']  = df['NAICS'].astype(str).str.zfill(6)
    return df

from typing import Union, List

def compute_effective_tariff_rates(
    df,
    dates,  # str or list of str, e.g. '2025-07' or ['2025-04', '2025-07']
):
    if isinstance(dates, str):
        dates = [dates]

    date_list = [pd.to_datetime(d) for d in dates]

    mask = df['time'].isin(date_list)
    subset = df[mask].copy()

    if subset.empty:
        available = df['time'].dt.strftime('%Y-%m').unique()
        raise ValueError(
            f"No data found for {dates}. "
            f"Available range: {sorted(available)[0]} to {sorted(available)[-1]}"
        )

    result = (
        subset
        .groupby(['naics6', 'NAICS_SDESC', 'time'])[['duties', 'imports']]
        .sum()
        .reset_index()
        .assign(tau=lambda x: x['duties'] / x['imports'].replace(0, float('nan')))
        .sort_values(['time', 'naics6'])
        .reset_index(drop=True)
    )

    found_dates = result['time'].dt.strftime('%Y-%m').unique()
    missing = [d for d in dates if d not in found_dates]
    if missing:
        print(f"Warning: no data found for dates: {missing}")

    return result[['naics6', 'NAICS_SDESC', 'time', 'imports', 'duties', 'tau']]

In [21]:
df = load_naics_imports('./data/imports/TOTALnaics-data-2025-12.parquet')

rates_july = compute_effective_tariff_rates(df, '2025-07')

In [22]:
rates_july.head()

,naics6,NAICS_SDESC,time,imports,duties,tau
0,111110,SOYBEANS,2025-07-01,61349578,3691443,0.060171
1,111120,OILSEEDS (EXCEPT SOYBEAN),2025-07-01,52020228,2966960,0.057035
2,111130,DRY PEAS & BEANS,2025-07-01,55359882,3498665,0.063199
3,111140,WHEAT,2025-07-01,54341944,4984,0.000092
4,111150,CORN,2025-07-01,14567765,673805,0.046253
